# Notebook 03 — ETL y Limpieza de Datos

**Proyecto:** Sistema de Predicción y Clasificación de la Desnutrición en niños menores de cinco años  
**Fuente:** SIVIGILA — Evento 113 (Desnutrición aguda) — 2023, 2024, 2025  

---
## Contenido
1. Importación de librerías
2. Carga de datos desde Excel originales (126 columnas)
3. Eliminación de datos personales (privacidad)
4. Diagnóstico inicial de nulos
5. Filtro de edad — menores de 5 años
6. Corrección de tipos — decimales con coma (formato es-CO)
7. Reemplazo de ceros por NaN
8. Tratamiento de outliers (criterios OMS)
9. Conversión a entero de columnas categóricas
10. Revisión de nulos y rangos post-filtros
11. Validación y conversión de fechas
12. Variables temporales derivadas
13. Validación coherencia edad vs. fecha de nacimiento
14. Limpieza de variables categóricas
15. Imputación de valores nulos
16. Creación de columnas derivadas y etiquetas legibles
17. Selección óptima de variables
18. Dataset final — corrección final de tipos
19. Resumen final por dataset
20. Unión de los tres datasets
21. Resumen del dataset unificado
22. Exportación de archivos de salida

## 1. Importación de librerías

In [31]:
import sys
!{sys.executable} -m pip install openpyxl --quiet

import pandas as pd
import numpy as np
import os
import warnings

warnings.filterwarnings('ignore')
pd.set_option('display.max_columns', None)
pd.set_option('display.float_format', '{:.2f}'.format)

print("Librerías cargadas correctamente.")

Librerías cargadas correctamente.


## 2. Carga de datos desde Excel originales

Se cargan los 3 archivos Excel directamente del SIVIGILA (evento 113).  
Cada archivo tiene **126 columnas** originales, incluyendo fechas, variables clínicas  
y socioeconómicas que el CSV unificado anterior había perdido.

In [32]:
data2023 = pd.read_excel("../data/raw/2023_113_Reporte.xlsx")
data2024 = pd.read_excel("../data/raw/2024_113_Reporte.xlsx")
data2025 = pd.read_excel("../data/raw/2025_113_Reporte.xlsx")

datasets = {
    "2023": data2023,
    "2024": data2024,
    "2025": data2025,
}

for year, df in datasets.items():
    print(f"{year}: {df.shape[0]:,} filas × {df.shape[1]} columnas")

2023: 107 filas × 126 columnas
2024: 1,441 filas × 126 columnas
2025: 1,057 filas × 126 columnas


## 3. Eliminación de datos personales

Se eliminan todas las columnas que permiten identificar directamente a pacientes o sus madres,
así como columnas administrativas irrelevantes para el análisis.
Esta es la **primera operación** antes de cualquier exploración.

In [33]:
cols_privados = [
    # Nombres y documentos del paciente
    "pri_nom_", "seg_nom_", "pri_ape_", "seg_ape_",
    "tip_ide_", "num_ide_", "telefono_",
    # Nombres y documentos de la madre
    "pri_nom_ma", "seg_nom_ma", "pri_ape_ma", "seg_ape_ma",
    "tip_ide_ma", "num_ide_ma",
    # Dirección y geolocalización exacta
    "bar_ver_", "dir_res_", "lat_dir", "long_dir",
    "localidad_", "cen_pobla_", "vereda_",
    # Identidad de género y orientación sexual (no relevante para el estudio)
    "otra_ident", "otra_orien", "iden_gener", "orient_sex",
    # Administrativos / trazabilidad interna SIVIGILA
    "cod_pre", "cod_sub", "cod_eve", "nom_eve", "nom_upgd", "nit_upgd",
    "uni_modif", "nuni_modif", "fec_arc_xl", "nom_dil_f_", "tel_dil_f_", "fec_aju_",
    "fm_fuerza", "fm_unidad", "fm_grado",
    "RegIniFec", "vacio", "cer_def_", "cbmte_", "res_pr_ape",
    "estrato_datos_complementarios", "ajuste_", "version", "cod_ase_",
    "sem_ges_",   # duplicado de edad_ges
]

for year, df in datasets.items():
    antes = df.shape[1]
    datasets[year] = df.drop(columns=[c for c in cols_privados if c in df.columns], errors="ignore")
    print(f"{year}: {antes} → {datasets[year].shape[1]} columnas (eliminadas: {antes - datasets[year].shape[1]})")

2023: 126 → 77 columnas (eliminadas: 49)
2024: 126 → 77 columnas (eliminadas: 49)
2025: 126 → 77 columnas (eliminadas: 49)


## 4. Diagnóstico inicial de valores nulos

Se revisa el estado de nulos **antes de cualquier transformación** para tener
una línea base que justifique las decisiones de imputación posteriores.

In [34]:
for year, df in datasets.items():
    print(f"\n=== {year} — Top 15 columnas con más nulos ===")
    nulos = df.isnull().sum().sort_values(ascending=False)
    nulos_pct = (nulos / len(df) * 100).round(1)
    resumen = pd.DataFrame({"nulos": nulos, "%": nulos_pct})
    print(resumen[resumen["nulos"] > 0].head(15).to_string())


=== 2023 — Top 15 columnas con más nulos ===
            nulos     %
peso_nac       54 50.50
diag_medic     44 41.10
tipo_manej     40 37.40
edad_ges       33 30.80
nom_grupo_     10  9.30
estrato_        3  2.80
gp_mad_com      3  2.80
gp_gestan       3  2.80
niv_educat      2  1.90

=== 2024 — Top 15 columnas con más nulos ===
                 nulos     %
peso_nac           830 57.60
edad_ges           447 31.00
nom_grupo_         127  8.80
tipo_manej          82  5.70
diag_medic          61  4.20
estrato_            28  1.90
gp_gestan           20  1.40
gp_mad_com          20  1.40
niv_educat          19  1.30
ocupacion_          14  1.00
clas_peso           13  0.90
clas_talla          11  0.80
cambios_cabello      1  0.10
crec_dllo            1  0.10
e_complem            1  0.10

=== 2025 — Top 15 columnas con más nulos ===
            nulos     %
peso_nac      534 50.50
edad_ges      328 31.00
nom_grupo_     97  9.20
tipo_manej     64  6.10
diag_medic     62  5.90
estrato_      

## 5. Filtro de edad — menores de 5 años

El evento 113 aplica exclusivamente a menores de 5 años.
- `uni_med_ = 1` → edad en años: se conservan si `edad_ <= 5`
- `uni_med_ = 2` → edad en meses: siempre < 5 años, se conservan todos

In [35]:
for year, df in datasets.items():
    if "edad_" in df.columns and "uni_med_" in df.columns:
        antes = len(df)
        mask = (df["uni_med_"] == 2) | ((df["uni_med_"] == 1) & (df["edad_"] <= 5))
        datasets[year] = df[mask].copy()
        print(f"{year}: {len(datasets[year]):,} filas conservadas (eliminadas: {antes - len(datasets[year])})")

2023: 106 filas conservadas (eliminadas: 1)
2024: 1,432 filas conservadas (eliminadas: 9)
2025: 1,049 filas conservadas (eliminadas: 8)


## 6. Corrección de tipos de datos — decimales con coma (formato es-CO)

Varias columnas numéricas vienen como texto con **coma como separador decimal**
(formato colombiano: `14,70`, `-2,1066`, `7,4`). Esto hace que pandas las lea como
`object` en lugar de `float64`.

El patrón correcto es:
1. `.astype(str).str.strip()` — homogeneiza a texto
2. `.str.replace(',', '.', regex=False)` — convierte coma decimal a punto
3. `pd.to_numeric(..., errors='coerce')` — convierte a float; deja NaN si hay texto no numérico

**No** usar este proceso en columnas que ya son `float64` (ej. `peso_nac`, `edad_ges`).

In [36]:
# talla_nac también llega como str con coma decimal (ej. '49,5') y mezcla enteros ('40') y floats ('40.0').
cols_coma_decimal = ["peso_act", "talla_act", "talla_nac", "per_braqui", "imc", "zscore_pt", "zscore_te"]

for year, df in datasets.items():
    for col in cols_coma_decimal:
        if col in df.columns:
            # Paso 1: str para homogeneizar (enteros puros y decimales con coma conviven)
            # Paso 2: sustituir coma por punto
            # Paso 3: convertir a float — errors='coerce' deja NaN si hay texto no convertible
            df[col] = pd.to_numeric(
                df[col].astype(str).str.strip().str.replace(",", ".", regex=False),
                errors="coerce"
            )
    datasets[year] = df

# Verificación de tipos resultantes
print("Tipos de datos resultantes (2024):")
cols_ok = [c for c in cols_coma_decimal if c in datasets["2024"].columns]
print(datasets["2024"][cols_ok].dtypes)

Tipos de datos resultantes (2024):
peso_act      float64
talla_act     float64
talla_nac     float64
per_braqui    float64
imc           float64
zscore_pt     float64
zscore_te     float64
dtype: object


## 7. Reemplazo de ceros por NaN (valores no registrados)

En la ficha SIVIGILA, los campos numéricos no diligenciados quedan en `0`,
pero `0` es un valor clínicamente imposible en estas medidas:

- `peso_nac`: válido 900–5000 g → `0` = no registrado
- `talla_nac`: válido 30–55 cm
- `per_braqui`: válido 6–30 cm
- `edad_ges`: válido 20–45 semanas

In [37]:
cols_cero_es_nulo = ["peso_nac", "talla_nac", "per_braqui", "edad_ges"]

for year, df in datasets.items():
    for col in cols_cero_es_nulo:
        if col in df.columns:
            n = (df[col] == 0).sum()
            df[col] = df[col].replace(0, float("nan"))
            if n > 0:
                print(f"{year} | {col}: {n} ceros → NaN")
    datasets[year] = df

2023 | peso_nac: 17 ceros → NaN
2023 | talla_nac: 71 ceros → NaN
2023 | per_braqui: 18 ceros → NaN
2023 | edad_ges: 6 ceros → NaN
2024 | peso_nac: 12 ceros → NaN
2024 | talla_nac: 913 ceros → NaN
2024 | per_braqui: 116 ceros → NaN
2024 | edad_ges: 6 ceros → NaN
2025 | peso_nac: 22 ceros → NaN
2025 | talla_nac: 603 ceros → NaN
2025 | per_braqui: 138 ceros → NaN
2025 | edad_ges: 7 ceros → NaN


## 8. Tratamiento de outliers

Rangos válidos según la ficha de notificación y criterios clínicos para menores de 5 años.

#### Peso actual (1 – 25 kg)

In [38]:
for year, df in datasets.items():
    if "peso_act" in df.columns:
        antes = len(df)
        datasets[year] = df[(df["peso_act"] >= 1) & (df["peso_act"] <= 25)]
        print(f"{year}: {antes - len(datasets[year])} filas eliminadas por peso_act fuera de [1, 25] kg")

2023: 1 filas eliminadas por peso_act fuera de [1, 25] kg
2024: 3 filas eliminadas por peso_act fuera de [1, 25] kg
2025: 3 filas eliminadas por peso_act fuera de [1, 25] kg


#### Talla actual (45 – 150 cm)

In [39]:
for year, df in datasets.items():
    if "talla_act" in df.columns:
        antes = len(df)
        datasets[year] = df[(df["talla_act"] >= 45) & (df["talla_act"] <= 150)]
        print(f"{year}: {antes - len(datasets[year])} filas eliminadas por talla_act fuera de [45, 150] cm")

2023: 0 filas eliminadas por talla_act fuera de [45, 150] cm
2024: 0 filas eliminadas por talla_act fuera de [45, 150] cm
2025: 0 filas eliminadas por talla_act fuera de [45, 150] cm


#### IMC (10 – 30 kg/m²)

In [40]:
for year, df in datasets.items():
    if "imc" in df.columns:
        antes = len(df)
        datasets[year] = df[(df["imc"] >= 10) & (df["imc"] <= 30)]
        print(f"{year}: {antes - len(datasets[year])} filas eliminadas por imc fuera de [10, 30]")

2023: 1 filas eliminadas por imc fuera de [10, 30]
2024: 16 filas eliminadas por imc fuera de [10, 30]
2025: 10 filas eliminadas por imc fuera de [10, 30]


#### Z-score peso/talla — valores imposibles (< −6 o > 5)

In [41]:
for year, df in datasets.items():
    if "zscore_pt" in df.columns:
        antes = len(df)
        datasets[year] = df[(df["zscore_pt"] >= -6) & (df["zscore_pt"] <= 5)]
        print(f"{year}: {antes - len(datasets[year])} filas eliminadas por zscore_pt fuera de [-6, 5]")

2023: 0 filas eliminadas por zscore_pt fuera de [-6, 5]
2024: 1 filas eliminadas por zscore_pt fuera de [-6, 5]
2025: 4 filas eliminadas por zscore_pt fuera de [-6, 5]


#### Z-score talla/edad — valores imposibles (< −6 o > 6)

In [42]:
# La OMS establece que z-scores fuera de [-6, 6] indican error de medición.
# zscore_pt ya se filtró arriba; zscore_te requiere el mismo tratamiento.
for year, df in datasets.items():
    if "zscore_te" in df.columns:
        antes = len(df)
        datasets[year] = df[(df["zscore_te"] >= -6) & (df["zscore_te"] <= 6)]
        print(f"{year}: {antes - len(datasets[year])} filas eliminadas por zscore_te fuera de [-6, 6]")

2023: 6 filas eliminadas por zscore_te fuera de [-6, 6]
2024: 48 filas eliminadas por zscore_te fuera de [-6, 6]
2025: 23 filas eliminadas por zscore_te fuera de [-6, 6]


#### Error en clasificación (`clas_peso = 7`)

In [43]:
for year, df in datasets.items():
    if "clas_peso" in df.columns:
        antes = len(df)
        datasets[year] = df[df["clas_peso"] != 7]
        print(f"{year}: {antes - len(datasets[year])} filas eliminadas por clas_peso == 7 (error de codificación)")

2023: 0 filas eliminadas por clas_peso == 7 (error de codificación)
2024: 2 filas eliminadas por clas_peso == 7 (error de codificación)
2025: 0 filas eliminadas por clas_peso == 7 (error de codificación)


## 9. Conversión a entero de columnas categóricas

Estas columnas son enteras en la ficha pero pandas las lee como `float64` porque el Excel
tiene celdas vacías que introducen NaN. Una vez eliminadas las filas inválidas ya no tienen
NaN y se pueden castear a `Int64` (entero nullable de pandas). Esto evita que aparezcan
como `1.0` / `2.0` en lugar de `1` / `2`.

In [44]:
cols_enteras = [
    "clas_peso",        # 1=Desnut. severa, 2=Desnut. moderada, 3=Normal bajo, 4=Normal, 5=Sobrepeso, 6=Obesidad
    "clas_talla",       # 1=Normal, 2=Riesgo baja talla, 3=Baja talla
    "edema",            # 1=Sí, 2=No
    "delgadez",         # 1=Sí, 2=No
    "palidez",          # 1=Sí, 2=No
    "piel_rese",        # 1=Sí, 2=No
    "hiperpigm",        # 1=Sí, 2=No
    "cambios_cabello",  # 1=Sí, 2=No
    "crec_dllo",        # 1=Con seguimiento C&D, 2=Sin seguimiento C&D
    "esq_vac",          # 1=Completo, 2=Incompleto, 3=Sin esquema
    "carne_vac",        # 1=Sí, 2=No
    "ruta_atenc",       # 1=Urgencias, 2=Consulta externa
    "tipo_manej",       # 1=Ambulatorio, 2=Hospitalario
    "pac_hos_",         # 1=Sí, 2=No
]

for year, df in datasets.items():
    for col in cols_enteras:
        if col in df.columns:
            df[col] = pd.to_numeric(df[col], errors="coerce").astype("Int64")
    datasets[year] = df

for year, df in datasets.items():
    cols_pres = [c for c in cols_enteras if c in df.columns]
    print(f"\n=== {year} — tipos columnas enteras ===")
    print(df[cols_pres].dtypes)
    print("NaN clas_peso:", df["clas_peso"].isna().sum() if "clas_peso" in df.columns else "N/A")


=== 2023 — tipos columnas enteras ===
clas_peso          Int64
clas_talla         Int64
edema              Int64
delgadez           Int64
palidez            Int64
piel_rese          Int64
hiperpigm          Int64
cambios_cabello    Int64
crec_dllo          Int64
esq_vac            Int64
carne_vac          Int64
ruta_atenc         Int64
tipo_manej         Int64
pac_hos_           Int64
dtype: object
NaN clas_peso: 0

=== 2024 — tipos columnas enteras ===
clas_peso          Int64
clas_talla         Int64
edema              Int64
delgadez           Int64
palidez            Int64
piel_rese          Int64
hiperpigm          Int64
cambios_cabello    Int64
crec_dllo          Int64
esq_vac            Int64
carne_vac          Int64
ruta_atenc         Int64
tipo_manej         Int64
pac_hos_           Int64
dtype: object
NaN clas_peso: 9

=== 2025 — tipos columnas enteras ===
clas_peso          Int64
clas_talla         Int64
edema              Int64
delgadez           Int64
palidez            In

## 10. Revisión de nulos y rangos post-filtros

Verificación intermedia para confirmar que los outliers y ceros fueron correctamente
eliminados o marcados como NaN.

In [45]:
cols_check = ["peso_act", "talla_act", "talla_nac", "per_braqui", "imc", "zscore_pt", "zscore_te"]
for year, df in datasets.items():
    cols_presentes = [c for c in cols_check if c in df.columns]
    print(f"\n=== {year} ===")
    print(df[cols_presentes].isnull().sum().rename("NaN"))
    print(df[cols_presentes].agg(["min", "max"]))


=== 2023 ===
peso_act       0
talla_act      0
talla_nac     65
per_braqui    17
imc            0
zscore_pt      0
zscore_te      0
Name: NaN, dtype: int64
     peso_act  talla_act  talla_nac  per_braqui   imc  zscore_pt  zscore_te
min      2.30      48.00      40.00        6.00 10.00      -5.28      -5.96
max     18.50      98.50      55.00       14.50 19.07       2.52       4.26

=== 2024 ===
peso_act        0
talla_act       0
talla_nac     875
per_braqui     95
imc             0
zscore_pt       0
zscore_te       0
Name: NaN, dtype: int64
     peso_act  talla_act  talla_nac  per_braqui   imc  zscore_pt  zscore_te
min      2.40      48.00      30.00        6.00 10.00      -5.96      -6.00
max     16.50     104.50      55.00       18.00 21.90       2.89       4.45

=== 2025 ===
peso_act        0
talla_act       0
talla_nac     584
per_braqui    128
imc             0
zscore_pt       0
zscore_te       0
Name: NaN, dtype: int64
     peso_act  talla_act  talla_nac  per_braqui   imc  zsco

## 11. Validación y conversión de fechas

Se convierten todas las columnas de fecha al tipo `datetime64`.
El formato de la ficha SIVIGILA es `dd/mm/aaaa` → `dayfirst=True`.

In [46]:
cols_fecha = [
    "fec_not", "fecha_nto_", "FechaHora",
    "ini_sin_", "fec_con_", "fec_def_", "fec_hos_"
]

for year, df in datasets.items():
    for col in cols_fecha:
        if col in df.columns:
            # dayfirst=True porque el formato de la ficha es dd/mm/aaaa
            df[col] = pd.to_datetime(df[col], dayfirst=True, errors="coerce")
    datasets[year] = df
    print(f"{year}: fechas convertidas")

2023: fechas convertidas


2024: fechas convertidas
2025: fechas convertidas


## 12. Variables temporales derivadas

A partir de `fec_not` se generan variables que permiten el análisis de series temporales:
- `mes` — número de mes (1–12) para análisis estacional
- `anio_mes` — período año-mes en formato `'YYYY-MM'` para la serie temporal mensual

In [47]:
for year, df in datasets.items():
    if "fec_not" in df.columns:
        df["mes"] = df["fec_not"].dt.month
        df["anio_mes"] = df["fec_not"].dt.to_period("M").astype(str)
    datasets[year] = df

print("Variables temporales creadas: mes, anio_mes")
print("\nDistribución mensual 2024:")
print(datasets["2024"]["mes"].value_counts().sort_index())

Variables temporales creadas: mes, anio_mes

Distribución mensual 2024:
mes
1      93
2     108
3      83
4     101
5     123
6     130
7     168
8     141
9     120
10    116
11    102
12     77
Name: count, dtype: int64


## 13. Validación de coherencia: edad registrada vs. fecha de nacimiento

Se calcula la edad teórica a partir de `fec_not − fecha_nto_` y se compara
con la edad registrada. Registros con diferencia mayor a 1 año se eliminan
como incoherentes (posible error de digitación).

In [48]:
for year, df in datasets.items():
    requeridas = ["fec_not", "fecha_nto_", "edad_", "uni_med_"]
    if all(c in df.columns for c in requeridas):
        df = df.copy()
        df["_edad_calc_anios"] = (df["fec_not"] - df["fecha_nto_"]).dt.days / 365
        # Convertir edad registrada a años para comparar
        df["_edad_anios"] = df["edad_"].where(df["uni_med_"] == 1, df["edad_"] / 12)
        antes = len(df)
        df = df[abs(df["_edad_anios"] - df["_edad_calc_anios"]) < 1]
        df = df.drop(columns=["_edad_calc_anios", "_edad_anios"])
        datasets[year] = df
        print(f"{year}: {antes - len(df)} filas eliminadas por inconsistencia edad/fecha")
    else:
        print(f"{year}: columnas de fecha/edad no disponibles, validación omitida")

2023: 1 filas eliminadas por inconsistencia edad/fecha
2024: 12 filas eliminadas por inconsistencia edad/fecha
2025: 8 filas eliminadas por inconsistencia edad/fecha


## 14. Limpieza de variables categóricas

In [49]:
for year, df in datasets.items():
    if "sexo_" in df.columns:
        df["sexo_"] = df["sexo_"].astype(str).str.upper().str.strip()
        df["sexo_"] = df["sexo_"].replace({"MASCULINO": "M", "FEMENINO": "F"})
    datasets[year] = df
    vals = datasets[year]["sexo_"].unique() if "sexo_" in datasets[year].columns else "N/A"
    print(f"{year}: sexo_ limpiado — valores únicos: {vals}")

2023: sexo_ limpiado — valores únicos: ['M' 'F']
2024: sexo_ limpiado — valores únicos: ['F' 'M']
2025: sexo_ limpiado — valores únicos: ['F' 'M']


## 15. Imputación de valores nulos

In [50]:
for year, df in datasets.items():
    # peso_nac: imputar por mediana del grupo (edad_ges) para reducir el sesgo
    # de la mediana global que deja 33% de filas con el mismo valor artificial.
    if "peso_nac" in df.columns:
        if "edad_ges" in df.columns:
            df["peso_nac"] = df.groupby("edad_ges")["peso_nac"].transform(
                lambda x: x.fillna(x.median())
            )
        # Fallback: si quedaron NaN sin grupo, usar mediana global
        df["peso_nac"] = df["peso_nac"].fillna(df["peso_nac"].median())
    # edad_ges: imputar con mediana (ya no tiene ceros tras el paso anterior)
    if "edad_ges" in df.columns:
        df["edad_ges"] = df["edad_ges"].fillna(df["edad_ges"].median())
    # estrato_: usar 0 (entero) en lugar de 'Desconocido' para mantener tipo numerico
    if "estrato_" in df.columns:
        df["estrato_"] = pd.to_numeric(df["estrato_"], errors="coerce").fillna(0).astype(int)
    datasets[year] = df
    print(f"{year}: imputación completada")

2023: imputación completada
2024: imputación completada
2025: imputación completada


## 16. Creación de columnas derivadas

Se crean columnas con etiquetas legibles a partir de los códigos numéricos/geográficos
para facilitar el análisis descriptivo y la visualización.

In [51]:
# ─── 1. per_etn_ → etnia (etiqueta legible) ───────────────────────────────
mapa_etnia = {
    1: "Indigena",
    2: "Rom, Gitano",
    3: "Raizal",
    4: "Palenquero",
    5: "Negro, mulato, afrocolombiano",
    6: "Otro",
}

# ─── 2. estrato_: float → int, NaN/Desconocido → moda ────────────────────
# (operacion por año para que la moda sea representativa de cada dataset)

# ─── 3. cod_pais_o → pais_origen ─────────────────────────────────────────
mapa_pais = {
    170: "Colombia",
    862: "Venezuela",
}

# ─── 4. cod_dpto_o → depto_origen ────────────────────────────────────────
mapa_dpto = {
    "44": "La Guajira",
    "08": "Atlantico",
    "11": "Bogota D.C.",
    "13": "Bolivar",
    "20": "Cesar",
    "47": "Magdalena",
    "54": "Norte de Santander",
    "70": "Sucre",
    "D0": "Extranjero",   # codigo SIVIGILA para pacientes extranjeros
    "25": "Cundinamarca",
    "15": "Boyaca",
    "52": "Narino",
    "68": "Santander",
}

# ─── 5. cod_mun_o → municipio_origen (municipios del Cesar) ──────────────
mapa_municipio = {
    1:   "Valledupar",
    13:  "Agustin Codazzi",
    32:  "Astrea",
    45:  "Becerril",
    60:  "Bosconia",
    175: "Chimichagua",
    178: "Chiriquana",
    228: "Curumani",
    238: "El Copey",
    250: "El Paso",
    295: "Gamarra",
    383: "La Gloria",
    400: "La Jagua de Ibirico",
    517: "Pailitas",
    550: "Pelaya",
    570: "Pueblo Bello",
    614: "Rio de Oro",
    621: "La Paz (Robles)",
    710: "San Alberto",
    750: "San Diego",
    770: "San Martin",
    787: "Tamalameque",
    850: "Manaure Balcon del Cesar",
    873: "Gonzalez",

    # Municipios de La Guajira, Magdalena y otros departamentos presentes en los datos
    35:  "Albania",
    53:  "Aracataca",
    78:  "Barrancas",
    90:  "Dibulla",
    98:  "Distraccion",
    110: "El Molino",
    189: "Cienaga",
    279: "Fonseca",
    288: "Fundacion",
    378: "Hatonuevo",
    430: "Maicao",
    464: "Momil",
    560: "Manaure",
    650: "San Juan del Cesar",
    660: "Sabanas de San Angel",
    798: "Tenerife",
    835: "Turmeque",
    847: "Uribia",
    874: "Villanueva",

    # Municipios de otros departamentos pueden quedar como codigo numerico
    # si no estan en el mapa
}

for year, df in datasets.items():
    df = df.copy()

    # 1. Columna derivada de per_etn_
    if "per_etn_" in df.columns:
        df["etnia"] = df["per_etn_"].map(mapa_etnia)

    # 1b. nom_grupo_: nombre del grupo etnico especifico (ej. "Arhuaco", "Wayuu")
    #     La columna ya existe en el Excel original con el nombre del grupo.
    #     Se limpia y normaliza (strip + title case) para estandarizar valores.
    if "nom_grupo_" in df.columns:
        df["nom_grupo_"] = (
            df["nom_grupo_"]
            .astype(str)
            .str.strip()
            .str.title()
            .replace({"Nan": pd.NA, "None": pd.NA, "": pd.NA})
        )

    # 2. estrato_: float → int, reemplazar NaN por moda
    if "estrato_" in df.columns:
        # Convertir a numerico (elimina 'Desconocido' si quedara como str)
        df["estrato_"] = pd.to_numeric(df["estrato_"], errors="coerce")
        moda_estrato = int(df["estrato_"].mode()[0])
        df["estrato_"] = df["estrato_"].fillna(moda_estrato).astype(int)
        print(f"{year} | estrato_: moda = {moda_estrato}")

    # 3. Columna derivada de cod_pais_o
    if "cod_pais_o" in df.columns:
        df["pais_origen"] = df["cod_pais_o"].map(mapa_pais).fillna("Otro")

    # 4. Columna derivada de cod_dpto_o
    if "cod_dpto_o" in df.columns:
        # cod_dpto_o puede venir como int o str segun el año → normalizar con zfill(2)
        dpto_str = df["cod_dpto_o"].astype(str).str.strip().str.zfill(2)
        df["depto_origen"] = dpto_str.map(mapa_dpto).fillna("Otro departamento")

    # 5. Columna derivada de cod_mun_o
    if "cod_mun_o" in df.columns:
        df["municipio_origen"] = df["cod_mun_o"].map(mapa_municipio).fillna(
            df["cod_mun_o"].astype(str)   # fallback: dejar el codigo numerico como str
        )

    datasets[year] = df

# Verificacion rapida de columnas derivadas
df_muestra = datasets[list(datasets.keys())[0]]
cols_nuevas = [c for c in ["etnia", "nom_grupo_", "estrato_", "pais_origen", "depto_origen", "municipio_origen"] if c in df_muestra.columns]
print("\nColumnas derivadas creadas:", cols_nuevas)
for col in cols_nuevas:
    print(f"\n{col}:")
    print(df_muestra[col].value_counts().sort_index().to_string())

2023 | estrato_: moda = 1
2024 | estrato_: moda = 1
2025 | estrato_: moda = 1

Columnas derivadas creadas: ['etnia', 'nom_grupo_', 'estrato_', 'pais_origen', 'depto_origen', 'municipio_origen']

etnia:
etnia
Indigena    88
Otro         9

nom_grupo_:
nom_grupo_
Arhuaco    18
Kogui       2
Wayuu      43
Wiwa        2
Yukpa      22

estrato_:
estrato_
0     1
1    94
2     2

pais_origen:
pais_origen
Colombia    97

depto_origen:
depto_origen
Cesar         42
La Guajira    52
Magdalena      3

municipio_origen:
municipio_origen
Agustin Codazzi       20
Albania                2
Becerril               1
Dibulla                2
Maicao                 9
Manaure                9
Pueblo Bello          19
San Juan del Cesar     1
Uribia                18
Valledupar            15
Villanueva             1


## 17. Selección óptima de variables

In [52]:
demograficas = [
    "edad_", "uni_med_", "sexo_",
    "nacionali_", "per_etn_", "etnia",   # etnia: etiqueta legible de per_etn_
    "nom_grupo_",                         # nombre del grupo etnico especifico
    "estrato_",
]

geograficas = [
    "cod_pais_o", "pais_origen",          # pais_origen: etiqueta de cod_pais_o
    "cod_dpto_o", "depto_origen",         # depto_origen: nombre del departamento
    "cod_mun_o",  "municipio_origen",     # municipio_origen: nombre del municipio
    "area_",
    "cod_pais_r", "cod_dpto_r", "cod_mun_r",
    "ndep_resi", "nmun_resi",
]

temporales = [
    "fec_not", "semana", "mes", "anio_mes",
    "fecha_nto_", "fec_con_",
]

socioecon = [
    "tip_ss_",
    "niv_educat",
    "menores",
    "gp_desplaz", "gp_migrant", "gp_indigen",
    "gp_pobicbf", "gp_mad_com", "gp_vic_vio",
    "gp_gestan", "ocupacion_",
]

clinicas = [
    "peso_nac", "talla_nac", "edad_ges",
    "t_lechem", "e_complem",
    "crec_dllo", "esq_vac", "carne_vac",
    "peso_act", "talla_act", "per_braqui",
    "imc", "zscore_pt", "zscore_te",
    "clas_peso", "clas_talla",
    "edema", "delgadez", "palidez",
    "piel_rese", "hiperpigm", "cambios_cabello",
]

diagnostico = [
    "diag_medic", "ruta_atenc", "tipo_manej",
    "pac_hos_", "con_fin_", "tip_cas_", "fuente_",
]

cols_finales = demograficas + geograficas + temporales + socioecon + clinicas + diagnostico

for year, df in datasets.items():
    cols_disponibles = [c for c in cols_finales if c in df.columns]
    datasets[year] = df[cols_disponibles].copy()
    print(f"{year} - Shape final: {datasets[year].shape}")

data2023 = datasets["2023"]
data2024 = datasets["2024"]
data2025 = datasets["2025"]

2023 - Shape final: (97, 66)
2024 - Shape final: (1350, 66)
2025 - Shape final: (1001, 66)


## 18. Dataset final — corrección final de tipos

**1. `talla_nac`** — puede quedar como `object` con mezcla de enteros (`'40'`), decimales con coma (`'49,5'`) y ceros. Se convierte a `float64` y los ceros se marcan como `NaN`.

**2. Columnas categóricas** (`clas_peso`, `clas_talla`, `edema`, `delgadez`, `palidez`, `piel_rese`, `hiperpigm`, `cambios_cabello`) — quedan como `float64` con `.0` porque pandas usa float cuando hay NaN en columnas de enteros. Se convierten a `Int64` (entero nullable) para eliminar los `.0` sin perder los NaN.

In [53]:
# talla_nac: convertir a float64 y limpiar ceros residuales
for year, df in datasets.items():
    if "talla_nac" in df.columns:
        df["talla_nac"] = pd.to_numeric(
            df["talla_nac"].astype(str).str.strip().str.replace(",", ".", regex=False),
            errors="coerce"
        )
        n_ceros = (df["talla_nac"] == 0).sum()
        df["talla_nac"] = df["talla_nac"].replace(0, float("nan"))
        print(f"{year} | talla_nac: dtype={df['talla_nac'].dtype}, {n_ceros} ceros -> NaN, NaN total={df['talla_nac'].isna().sum()}")
    datasets[year] = df

2023 | talla_nac: dtype=float64, 0 ceros -> NaN, NaN total=65
2024 | talla_nac: dtype=float64, 0 ceros -> NaN, NaN total=867
2025 | talla_nac: dtype=float64, 0 ceros -> NaN, NaN total=578


In [54]:
# Columnas categoricas ordinales: float64 con .0 -> Int64 (entero nullable de pandas)
cols_categoricas_int = [
    "clas_peso", "clas_talla",
    "edema", "delgadez", "palidez",
    "piel_rese", "hiperpigm", "cambios_cabello"
]

for year, df in datasets.items():
    for col in cols_categoricas_int:
        if col in df.columns:
            df[col] = pd.to_numeric(df[col], errors="coerce").astype("Int64")
    datasets[year] = df
    cols_ok = [c for c in cols_categoricas_int if c in datasets[year].columns]
    print(f"{year}: categoricas -> {datasets[year][cols_ok].dtypes.unique()}",
          "| NaN clas_peso:", datasets[year]["clas_peso"].isna().sum() if "clas_peso" in datasets[year].columns else "N/A")

2023: categoricas -> [Int64Dtype()] | NaN clas_peso: 0
2024: categoricas -> [Int64Dtype()] | NaN clas_peso: 9
2025: categoricas -> [Int64Dtype()] | NaN clas_peso: 0


In [55]:
cols_finales_sel = demograficas + geograficas + temporales + socioecon + clinicas + diagnostico

for year, df in datasets.items():
    cols_disponibles = [c for c in cols_finales_sel if c in df.columns]
    datasets[year] = df[cols_disponibles].copy()
    print(f"{year} - Shape final: {datasets[year].shape}")

data2023 = datasets["2023"]
data2024 = datasets["2024"]
data2025 = datasets["2025"]

2023 - Shape final: (97, 66)
2024 - Shape final: (1350, 66)
2025 - Shape final: (1001, 66)


## 19. Resumen final de todos los datasets

In [56]:
for year, df in datasets.items():
    print(f"\n{'='*50}")
    print(f"  DATASET {year} - RESUMEN FINAL")
    print(f"{'='*50}")

    filas, columnas = df.shape
    print(f" Filas    : {filas}")
    print(f" Columnas : {columnas}")

    nulos_totales = df.isnull().sum().sum()
    pct = (nulos_totales / (filas * columnas) * 100) if filas * columnas > 0 else 0
    print(f" Nulos    : {nulos_totales} ({pct:.2f}%)")

    nulos_col = df.isnull().sum().sort_values(ascending=False)
    nulos_col = nulos_col[nulos_col > 0]
    if not nulos_col.empty:
        print(" Columnas con nulos remanentes:")
        print(nulos_col.to_string())
    else:
        print(" Sin valores nulos.")

    print("\n Tipos de datos:")
    print(df.dtypes.to_string())
    print("\n Estadisticas descriptivas:")
    display(df.describe())
    print("\n Primeras 5 filas:")
    display(df.head())


  DATASET 2023 - RESUMEN FINAL
 Filas    : 97
 Columnas : 66
 Nulos    : 180 (2.81%)
 Columnas con nulos remanentes:
talla_nac     65
diag_medic    42
tipo_manej    38
per_braqui    17
nom_grupo_    10
gp_mad_com     3
gp_gestan      3
niv_educat     2

 Tipos de datos:
edad_                        int64
uni_med_                     int64
sexo_                       object
nacionali_                   int64
per_etn_                     int64
etnia                       object
nom_grupo_                  object
estrato_                     int64
cod_pais_o                   int64
pais_origen                 object
cod_dpto_o                   int64
depto_origen                object
cod_mun_o                    int64
municipio_origen            object
area_                        int64
cod_pais_r                   int64
cod_dpto_r                   int64
cod_mun_r                    int64
ndep_resi                   object
nmun_resi                   object
fec_not             datetime

,edad_,uni_med_,nacionali_,per_etn_,estrato_,cod_pais_o,cod_dpto_o,cod_mun_o,area_,cod_pais_r,cod_dpto_r,cod_mun_r,fec_not,semana,mes,fecha_nto_,fec_con_,niv_educat,menores,gp_desplaz,gp_migrant,gp_indigen,gp_pobicbf,gp_mad_com,gp_vic_vio,gp_gestan,ocupacion_,peso_nac,talla_nac,edad_ges,t_lechem,e_complem,crec_dllo,esq_vac,carne_vac,peso_act,talla_act,per_braqui,imc,zscore_pt,zscore_te,clas_peso,clas_talla,edema,delgadez,palidez,piel_rese,hiperpigm,cambios_cabello,ruta_atenc,tipo_manej,pac_hos_,con_fin_,tip_cas_,fuente_
count,97.00,97.00,97.00,97.00,97.00,97.00,97.00,97.00,97.00,97.00,97.00,97.00,97,97.00,97.00,97,97,95.00,97.00,97.00,97.00,97.00,97.00,94.00,97.00,94.00,97.00,97.00,32.00,97.00,97.00,97.00,97.00,97.00,97.00,97.00,97.00,80.00,97.00,97.00,97.00,97.00,97.00,97.00,97.00,97.00,97.00,97.00,97.00,97.00,59.00,97.00,97.00,97.00,97.00
mean,2.78,1.27,170.00,1.46,1.01,170.00,33.70,382.27,2.61,170.00,34.20,378.20,2023-08-15 07:25:21.649484544,31.62,7.72,2022-01-30 10:08:39.587628800,2023-08-06 07:55:03.092783616,3.28,1.40,2.00,2.00,2.00,1.95,2.00,2.00,2.00,99999.07,2832.58,48.38,38.26,7.49,5.69,1.43,1.58,1.57,7.23,71.72,12.42,13.71,-2.20,-2.56,2.24,1.42,1.75,1.24,1.56,1.30,1.55,1.32,1.03,1.36,1.58,1.00,4.00,1.40
min,1.00,1.00,170.00,1.00,0.00,170.00,20.00,1.00,1.00,170.00,20.00,1.00,2023-01-10 00:00:00,2.00,1.00,2018-07-16 00:00:00,2023-01-10 00:00:00,1.00,0.00,2.00,2.00,2.00,1.00,2.00,2.00,2.00,99999.06,1700.00,40.00,25.00,0.00,0.00,1.00,1.00,1.00,2.30,48.00,6.00,10.00,-5.28,-5.96,1.00,1.00,1.00,1.00,1.00,1.00,1.00,1.00,1.00,1.00,1.00,1.00,4.00,1.00
25%,1.00,1.00,170.00,1.00,1.00,170.00,20.00,13.00,3.00,170.00,20.00,13.00,2023-05-29 00:00:00,21.00,5.00,2021-08-28 00:00:00,2023-05-26 00:00:00,1.00,1.00,2.00,2.00,2.00,2.00,2.00,2.00,2.00,99999.07,2900.00,47.00,38.00,3.00,3.00,1.00,1.00,1.00,6.15,66.50,12.00,13.10,-2.61,-3.72,2.00,1.00,2.00,1.00,1.00,1.00,1.00,1.00,1.00,1.00,1.00,1.00,4.00,1.00
50%,2.00,1.00,170.00,1.00,1.00,170.00,44.00,430.00,3.00,170.00,44.00,430.00,2023-08-18 00:00:00,33.00,8.00,2022-04-03 00:00:00,2023-08-15 00:00:00,5.00,1.00,2.00,2.00,2.00,2.00,2.00,2.00,2.00,99999.07,2900.00,49.00,39.00,6.00,6.00,1.00,1.00,2.00,7.10,72.00,12.50,13.67,-2.30,-2.91,2.00,1.00,2.00,1.00,2.00,1.00,2.00,1.00,1.00,1.00,2.00,1.00,4.00,1.00
75%,3.00,2.00,170.00,1.00,1.00,170.00,44.00,570.00,3.00,170.00,44.00,570.00,2023-11-24 00:00:00,46.00,11.00,2022-11-01 00:00:00,2023-11-14 00:00:00,5.00,2.00,2.00,2.00,2.00,2.00,2.00,2.00,2.00,99999.07,2900.00,50.00,39.00,12.00,6.00,2.00,2.00,2.00,8.20,77.00,13.00,14.07,-2.10,-1.87,2.00,2.00,2.00,1.00,2.00,2.00,2.00,2.00,1.00,2.00,2.00,1.00,4.00,1.00
max,11.00,2.00,170.00,6.00,2.00,170.00,47.00,874.00,3.00,170.00,68.00,874.00,2024-03-08 00:00:00,52.00,12.00,2023-10-07 00:00:00,2023-12-30 00:00:00,5.00,8.00,2.00,2.00,2.00,2.00,2.00,2.00,2.00,99999.07,3600.00,55.00,40.00,56.00,69.00,2.00,3.00,2.00,18.50,98.50,14.50,19.07,2.52,4.26,6.00,3.00,2.00,2.00,2.00,2.00,2.00,2.00,2.00,2.00,2.00,1.00,4.00,4.00
std,2.74,0.45,0.00,1.46,0.18,0.00,12.05,332.43,0.78,0.00,12.46,331.68,NaN,14.07,3.36,NaN,NaN,1.92,1.36,0.00,0.00,0.00,0.22,0.00,0.00,0.00,0.00,282.93,3.11,2.61,7.08,7.39,0.50,0.76,0.50,2.29,9.72,1.23,1.55,1.23,1.77,0.95,0.73,0.43,0.43,0.50,0.46,0.50,0.47,0.17,0.48,0.50,0.00,0.00,0.92



 Primeras 5 filas:


,edad_,uni_med_,sexo_,nacionali_,per_etn_,etnia,nom_grupo_,estrato_,cod_pais_o,pais_origen,cod_dpto_o,depto_origen,cod_mun_o,municipio_origen,area_,cod_pais_r,cod_dpto_r,cod_mun_r,ndep_resi,nmun_resi,fec_not,semana,mes,anio_mes,fecha_nto_,fec_con_,tip_ss_,niv_educat,menores,gp_desplaz,gp_migrant,gp_indigen,gp_pobicbf,gp_mad_com,gp_vic_vio,gp_gestan,ocupacion_,peso_nac,talla_nac,edad_ges,t_lechem,e_complem,crec_dllo,esq_vac,carne_vac,peso_act,talla_act,per_braqui,imc,zscore_pt,zscore_te,clas_peso,clas_talla,edema,delgadez,palidez,piel_rese,hiperpigm,cambios_cabello,diag_medic,ruta_atenc,tipo_manej,pac_hos_,con_fin_,tip_cas_,fuente_
0,6,2,M,170,1,Indigena,Wayuu,1,170,Colombia,44,La Guajira,847,Uribia,3,170,44,847,GUAJIRA,URIBIA,2023-01-10,2,1,2023-01,2022-06-29,2023-01-10,S,5.00,1,2,2,2,2,2.00,2,2.00,99999.07,2620.00,47.00,38.00,10,4,1,1,1,5.00,60.00,11.50,13.89,-2.24,-3.56,2,1,2,1,2,2,2,2,E440,1,2,2,1,4,1
2,1,1,M,170,1,Indigena,Wayuu,1,170,Colombia,44,La Guajira,847,Uribia,3,170,44,847,GUAJIRA,URIBIA,2023-05-08,19,5,2023-05,2022-04-13,2023-05-08,S,1.00,2,2,2,2,2,2.00,2,2.00,99999.07,2800.00,NaN,38.00,6,6,1,1,1,7.40,71.50,12.10,14.50,-2.11,-1.79,2,2,2,1,2,2,2,1,E440,1,1,1,1,4,1
3,10,2,M,170,1,Indigena,Arhuaco,1,170,Colombia,20,Cesar,570,Pueblo Bello,3,170,20,570,CESAR,PUEBLO BELLO,2023-10-04,40,10,2023-10,2022-11-24,2023-10-04,S,5.00,1,2,2,2,2,2.00,2,2.00,99999.07,3000.00,NaN,40.00,6,6,1,3,2,7.30,72.00,13.00,14.10,-2.44,-0.56,2,3,2,1,2,2,1,1,E440,1,2,2,1,4,1
4,1,1,F,170,6,Otro,<NA>,1,170,Colombia,20,Cesar,570,Pueblo Bello,3,170,20,570,CESAR,PUEBLO BELLO,2023-05-11,19,5,2023-05,2022-03-11,2023-05-11,S,1.00,1,2,2,2,2,2.00,2,2.00,99999.07,2900.00,NaN,39.00,6,6,2,1,2,7.30,70.50,6.00,14.70,-1.40,-2.19,3,1,1,1,1,1,1,1,E440,1,1,1,1,4,1
5,2,1,F,170,6,Otro,<NA>,2,170,Colombia,20,Cesar,570,Pueblo Bello,1,170,20,570,CESAR,PUEBLO BELLO,2023-03-08,10,3,2023-03,2021-02-16,2023-03-08,S,5.00,4,2,2,2,2,2.00,2,2.00,99999.07,2900.00,NaN,39.00,0,0,2,3,2,8.70,79.00,13.70,13.94,-1.63,-2.08,3,1,1,1,1,1,1,1,NaN,1,<NA>,1,1,4,1



  DATASET 2024 - RESUMEN FINAL
 Filas    : 1350
 Columnas : 66
 Nulos    : 1289 (1.45%)
 Columnas con nulos remanentes:
talla_nac     867
nom_grupo_    114
per_braqui     89
tipo_manej     77
diag_medic     58
gp_mad_com     19
gp_gestan      19
niv_educat     17
ocupacion_     13
clas_peso       9
clas_talla      7

 Tipos de datos:
edad_                        int64
uni_med_                     int64
sexo_                       object
nacionali_                   int64
per_etn_                     int64
etnia                       object
nom_grupo_                  object
estrato_                     int64
cod_pais_o                   int64
pais_origen                 object
cod_dpto_o                  object
depto_origen                object
cod_mun_o                    int64
municipio_origen            object
area_                        int64
cod_pais_r                   int64
cod_dpto_r                  object
cod_mun_r                    int64
ndep_resi                   objec

...

,edad_,uni_med_,sexo_,nacionali_,per_etn_,etnia,nom_grupo_,estrato_,cod_pais_o,pais_origen,cod_dpto_o,depto_origen,cod_mun_o,municipio_origen,area_,cod_pais_r,cod_dpto_r,cod_mun_r,ndep_resi,nmun_resi,fec_not,semana,mes,anio_mes,fecha_nto_,fec_con_,tip_ss_,niv_educat,menores,gp_desplaz,gp_migrant,gp_indigen,gp_pobicbf,gp_mad_com,gp_vic_vio,gp_gestan,ocupacion_,peso_nac,talla_nac,edad_ges,t_lechem,e_complem,crec_dllo,esq_vac,carne_vac,peso_act,talla_act,per_braqui,imc,zscore_pt,zscore_te,clas_peso,clas_talla,edema,delgadez,palidez,piel_rese,hiperpigm,cambios_cabello,diag_medic,ruta_atenc,tipo_manej,pac_hos_,con_fin_,tip_cas_,fuente_
0,1,1,F,170,1,Indigena,Yukpa,1,170,Colombia,20,Cesar,13,Agustin Codazzi,1,170,20,13,CESAR,AGUSTIN CODAZZI,2024-01-19,3,1,2024-01,2022-12-22,2024-01-17,S,5.00,1.00,2,2,2,2,2.00,2,2.00,9999.00,2500.00,49.00,39.00,6.00,12.00,1,3,2,6.50,69.00,12.00,13.65,-2.31,-1.95,2,2,2,1,1,1,2,1,NaN,1,<NA>,1,1,4,1
1,2,1,F,170,1,Indigena,Arhuaco,1,170,Colombia,20,Cesar,570,Pueblo Bello,1,170,20,570,CESAR,PUEBLO BELLO,2024-01-06,1,1,2024-01,2022-01-10,2024-01-06,S,5.00,3.00,2,2,2,2,2.00,2,2.00,99999.07,2845.00,NaN,39.00,0.00,0.00,2,2,2,7.80,70.00,13.00,15.92,-0.51,-4.89,4,1,1,1,1,1,1,1,NaN,1,<NA>,1,1,4,1
2,10,2,M,170,1,Indigena,Yukpa,1,170,Colombia,20,Cesar,13,Agustin Codazzi,3,170,20,13,CESAR,AGUSTIN CODAZZI,2024-03-15,11,3,2024-03,2023-04-27,2024-03-15,S,5.00,0.00,2,2,2,2,2.00,2,2.00,99999.07,2845.00,NaN,39.00,10.00,6.00,2,3,2,6.50,68.00,11.00,14.10,-2.57,-2.31,2,1,2,1,1,1,1,1,E440,1,1,1,1,4,1
3,1,1,M,170,1,Indigena,Wayuu,1,170,Colombia,44,La Guajira,847,Uribia,3,170,44,847,GUAJIRA,URIBIA,2024-01-15,3,1,2024-01,2022-09-04,2024-01-15,S,1.00,1.00,2,2,2,2,2.00,2,2.00,99999.07,2745.00,49.00,38.00,6.00,6.00,1,1,1,7.40,72.50,12.00,14.10,-2.42,-2.98,2,1,2,1,2,2,2,1,E440,1,2,2,1,4,1
4,1,1,F,170,1,Indigena,Yukpa,1,170,Colombia,20,Cesar,13,Agustin Codazzi,3,170,20,13,CESAR,AGUSTIN CODAZZI,2024-01-04,1,1,2024-01,2022-12-23,2024-01-03,S,5.00,2.00,2,2,2,2,2.00,2,2.00,99999.07,2845.00,NaN,39.00,12.00,6.00,2,3,2,8.10,72.00,NaN,15.60,-0.63,-0.78,4,3,1,2,2,1,1,1,E43X,1,1,1,1,4,1



  DATASET 2025 - RESUMEN FINAL
 Filas    : 1001
 Columnas : 66
 Nulos    : 967 (1.46%)
 Columnas con nulos remanentes:
talla_nac     578
per_braqui    122
nom_grupo_     90
tipo_manej     63
diag_medic     59
gp_mad_com     15
niv_educat     15
gp_gestan      15
gp_migrant      2
gp_pobicbf      2
gp_indigen      2
gp_desplaz      2
gp_vic_vio      2

 Tipos de datos:
edad_                        int64
uni_med_                     int64
sexo_                       object
nacionali_                   int64
per_etn_                     int64
etnia                       object
nom_grupo_                  object
estrato_                     int64
cod_pais_o                   int64
pais_origen                 object
cod_dpto_o                  object
depto_origen                object
cod_mun_o                    int64
municipio_origen            object
area_                        int64
cod_pais_r                   int64
cod_dpto_r                  object
cod_mun_r                    int6

,edad_,uni_med_,nacionali_,per_etn_,estrato_,cod_pais_o,cod_mun_o,area_,cod_pais_r,cod_mun_r,fec_not,semana,mes,fecha_nto_,fec_con_,niv_educat,menores,gp_desplaz,gp_migrant,gp_indigen,gp_pobicbf,gp_mad_com,gp_vic_vio,gp_gestan,ocupacion_,peso_nac,talla_nac,edad_ges,t_lechem,e_complem,crec_dllo,esq_vac,carne_vac,peso_act,talla_act,per_braqui,imc,zscore_pt,zscore_te,clas_peso,clas_talla,edema,delgadez,palidez,piel_rese,hiperpigm,cambios_cabello,ruta_atenc,tipo_manej,pac_hos_,con_fin_,tip_cas_,fuente_
count,1001.00,1001.00,1001.00,1001.00,1001.00,1001.00,1001.00,1001.00,1001.00,1001.00,1001,1001.00,1001.00,1001,1001,986.00,1001.00,999.00,999.00,999.00,999.00,986.00,999.00,986.00,1001.00,1001.00,423.00,1001.00,1001.00,1001.00,1001.00,1001.00,1001.00,1001.00,1001.00,879.00,1001.00,1001.00,1001.00,1001.00,1001.00,1001.00,1001.00,1001.00,1001.00,1001.00,1001.00,1001.00,938.00,1001.00,1001.00,1001.00,1001.00
mean,2.89,1.25,170.69,1.45,1.00,175.53,381.49,2.59,175.53,381.54,2025-06-26 15:06:17.622377472,26.11,6.41,2023-10-05 00:56:06.233766400,2025-06-25 20:19:54.005993984,3.21,1.28,2.00,2.00,2.00,1.98,2.00,2.00,2.00,98830.24,2739.16,48.13,38.38,10.24,5.72,1.48,1.80,1.64,7.47,73.34,12.37,13.56,-2.30,-2.73,2.09,1.40,1.79,1.30,1.62,1.46,1.67,1.41,1.07,1.43,1.59,1.00,4.00,1.23
min,1.00,1.00,170.00,1.00,0.00,170.00,1.00,1.00,170.00,1.00,2024-12-29 00:00:00,1.00,1.00,2020-04-14 00:00:00,2024-12-29 00:00:00,1.00,0.00,1.00,1.00,2.00,1.00,2.00,2.00,2.00,9999.00,970.00,35.00,24.00,0.00,0.00,1.00,1.00,1.00,2.30,47.00,8.00,10.00,-5.95,-5.99,1.00,1.00,1.00,1.00,1.00,1.00,1.00,1.00,1.00,1.00,1.00,1.00,4.00,1.00
25%,1.00,1.00,170.00,1.00,1.00,170.00,13.00,3.00,170.00,13.00,2025-03-29 00:00:00,13.00,3.00,2023-04-13 00:00:00,2025-03-29 00:00:00,1.00,1.00,2.00,2.00,2.00,2.00,2.00,2.00,2.00,99999.07,2750.00,47.00,38.00,6.00,6.00,1.00,1.00,1.00,6.20,67.50,11.70,13.00,-2.65,-3.64,2.00,1.00,2.00,1.00,1.00,1.00,1.00,1.00,1.00,1.00,1.00,1.00,4.00,1.00
50%,2.00,1.00,170.00,1.00,1.00,170.00,430.00,3.00,170.00,430.00,2025-06-24 00:00:00,26.00,6.00,2024-01-04 00:00:00,2025-06-24 00:00:00,3.00,1.00,2.00,2.00,2.00,2.00,2.00,2.00,2.00,99999.07,2800.00,49.00,39.00,9.00,6.00,1.00,2.00,2.00,7.40,73.00,12.20,13.54,-2.27,-2.79,2.00,1.00,2.00,1.00,2.00,1.00,2.00,1.00,1.00,1.00,2.00,1.00,4.00,1.00
75%,4.00,1.00,170.00,1.00,1.00,170.00,650.00,3.00,170.00,650.00,2025-09-25 00:00:00,39.00,9.00,2024-08-06 00:00:00,2025-09-24 00:00:00,5.00,2.00,2.00,2.00,2.00,2.00,2.00,2.00,2.00,99999.07,2800.00,50.00,39.00,14.00,6.00,2.00,3.00,2.00,8.60,78.90,13.00,14.10,-2.11,-1.84,2.00,2.00,2.00,2.00,2.00,2.00,2.00,2.00,1.00,2.00,2.00,1.00,4.00,1.00
max,11.00,2.00,862.00,6.00,3.00,862.00,980.00,3.00,862.00,980.00,2026-01-05 00:00:00,53.00,12.00,2025-11-09 00:00:00,2025-12-30 00:00:00,5.00,11.00,2.00,2.00,2.00,2.00,2.00,2.00,2.00,99999.07,4900.00,55.00,41.00,99.00,99.00,2.00,3.00,2.00,16.00,110.00,18.50,21.70,2.95,4.73,6.00,3.00,2.00,2.00,2.00,2.00,2.00,2.00,2.00,2.00,2.00,1.00,4.00,4.00
std,2.81,0.43,21.87,1.43,0.19,61.65,337.29,0.78,61.65,336.78,NaN,14.90,3.40,NaN,NaN,1.82,1.14,0.06,0.04,0.00,0.15,0.00,0.00,0.00,10194.74,337.42,3.34,1.77,7.54,4.82,0.50,0.87,0.48,2.27,10.97,1.21,1.36,0.99,1.48,0.79,0.69,0.41,0.46,0.49,0.50,0.47,0.49,0.26,0.50,0.49,0.00,0.00,0.74



 Primeras 5 filas:


,edad_,uni_med_,sexo_,nacionali_,per_etn_,etnia,nom_grupo_,estrato_,cod_pais_o,pais_origen,cod_dpto_o,depto_origen,cod_mun_o,municipio_origen,area_,cod_pais_r,cod_dpto_r,cod_mun_r,ndep_resi,nmun_resi,fec_not,semana,mes,anio_mes,fecha_nto_,fec_con_,tip_ss_,niv_educat,menores,gp_desplaz,gp_migrant,gp_indigen,gp_pobicbf,gp_mad_com,gp_vic_vio,gp_gestan,ocupacion_,peso_nac,talla_nac,edad_ges,t_lechem,e_complem,crec_dllo,esq_vac,carne_vac,peso_act,talla_act,per_braqui,imc,zscore_pt,zscore_te,clas_peso,clas_talla,edema,delgadez,palidez,piel_rese,hiperpigm,cambios_cabello,diag_medic,ruta_atenc,tipo_manej,pac_hos_,con_fin_,tip_cas_,fuente_
0,1,1,F,170,1,Indigena,Yukpa,1,170,Colombia,20,Cesar,13,Agustin Codazzi,3,170,20,13,CESAR,AGUSTIN CODAZZI,2025-01-11,2,1,2025-01,2023-11-27,2025-01-11,S,5.00,0,2.00,2.00,2.00,2.00,2.00,2.00,2.00,99999.07,2500.00,NaN,39.00,13,6,2,3,2,6.80,70.00,11.00,13.90,-2.09,-1.98,2,2,2,1,2,1,1,1,E440,1,1,1,1,4,1
1,7,2,F,170,1,Indigena,Wayuu,1,170,Colombia,44,La Guajira,847,Uribia,3,170,44,847,GUAJIRA,URIBIA,2025-01-09,2,1,2025-01,2024-05-28,2025-01-09,S,2.00,1,2.00,2.00,2.00,2.00,2.00,2.00,2.00,99999.07,2800.00,40.00,38.00,7,7,1,1,1,5.20,61.50,11.50,13.70,-2.09,-2.50,2,1,2,1,2,2,2,2,E440,1,2,2,1,4,1
2,1,1,M,170,1,Indigena,Wayuu,1,170,Colombia,44,La Guajira,847,Uribia,3,170,44,847,GUAJIRA,URIBIA,2025-01-10,2,1,2025-01,2023-07-19,2025-01-10,S,1.00,1,2.00,2.00,2.00,2.00,2.00,2.00,2.00,99999.07,2750.00,50.00,39.00,14,6,1,1,2,7.85,75.00,12.50,14.00,-2.38,-2.37,2,1,2,1,2,2,2,2,E440,1,2,2,1,4,1
3,2,1,F,170,1,Indigena,Wayuu,1,170,Colombia,44,La Guajira,847,Uribia,3,170,44,847,GUAJIRA,URIBIA,2025-01-05,2,1,2025-01,2022-08-01,2025-01-05,S,5.00,1,2.00,2.00,2.00,2.00,2.00,2.00,2.00,99999.07,2800.00,NaN,39.00,12,6,2,3,2,8.30,79.00,13.00,13.30,-2.19,-3.13,2,1,2,1,2,1,2,2,E440,2,2,2,1,4,1
4,1,1,F,170,1,Indigena,Wayuu,1,170,Colombia,44,La Guajira,847,Uribia,1,170,44,847,GUAJIRA,URIBIA,2025-01-10,2,1,2025-01,2023-07-31,2025-01-10,S,1.00,1,2.00,2.00,2.00,2.00,2.00,2.00,2.00,99999.07,3000.00,49.00,39.00,12,8,1,1,1,7.40,74.00,13.50,13.50,-2.20,-1.99,2,2,2,1,2,2,2,2,E440,1,2,2,1,4,1


## 20. Unión de los tres datasets en uno solo

In [57]:
# Agregar columna 'anio' para identificar el origen de cada registro
for year, df in datasets.items():
    datasets[year] = df.copy()
    datasets[year].insert(0, "anio", int(year))

# Concatenar los tres datasets
data_unificado = pd.concat(
    datasets.values(),
    ignore_index=True  # reindexar de 0 a N-1
)

print("Filas por anio en el dataset unificado:")
print(data_unificado["anio"].value_counts().sort_index())
print(f"\nShape total: {data_unificado.shape}")
print(f"Columnas   : {list(data_unificado.columns)}")

Filas por anio en el dataset unificado:
anio
2023      97
2024    1350
2025    1001
Name: count, dtype: int64

Shape total: (2448, 67)
Columnas   : ['anio', 'edad_', 'uni_med_', 'sexo_', 'nacionali_', 'per_etn_', 'etnia', 'nom_grupo_', 'estrato_', 'cod_pais_o', 'pais_origen', 'cod_dpto_o', 'depto_origen', 'cod_mun_o', 'municipio_origen', 'area_', 'cod_pais_r', 'cod_dpto_r', 'cod_mun_r', 'ndep_resi', 'nmun_resi', 'fec_not', 'semana', 'mes', 'anio_mes', 'fecha_nto_', 'fec_con_', 'tip_ss_', 'niv_educat', 'menores', 'gp_desplaz', 'gp_migrant', 'gp_indigen', 'gp_pobicbf', 'gp_mad_com', 'gp_vic_vio', 'gp_gestan', 'ocupacion_', 'peso_nac', 'talla_nac', 'edad_ges', 't_lechem', 'e_complem', 'crec_dllo', 'esq_vac', 'carne_vac', 'peso_act', 'talla_act', 'per_braqui', 'imc', 'zscore_pt', 'zscore_te', 'clas_peso', 'clas_talla', 'edema', 'delgadez', 'palidez', 'piel_rese', 'hiperpigm', 'cambios_cabello', 'diag_medic', 'ruta_atenc', 'tipo_manej', 'pac_hos_', 'con_fin_', 'tip_cas_', 'fuente_']


## 21. Resumen del dataset unificado

In [58]:
filas, columnas = data_unificado.shape
nulos_totales = data_unificado.isnull().sum().sum()
pct = (nulos_totales / (filas * columnas) * 100) if filas * columnas > 0 else 0

print(f" Filas    : {filas}")
print(f" Columnas : {columnas}")
print(f" Nulos    : {nulos_totales} ({pct:.2f}%)")

nulos_col = data_unificado.isnull().sum().sort_values(ascending=False)
nulos_col = nulos_col[nulos_col > 0]
if not nulos_col.empty:
    print("\n Columnas con nulos:")
    print(nulos_col.to_string())
else:
    print("\n Sin valores nulos.")

print("\n Tipos de datos:")
print(data_unificado.dtypes.to_string())
print("\n Estadisticas descriptivas:")
display(data_unificado.describe())
print("\n Primeras 5 filas:")
display(data_unificado.head())

 Filas    : 2448
 Columnas : 67
 Nulos    : 2436 (1.49%)

 Columnas con nulos:
talla_nac     1510
per_braqui     228
nom_grupo_     214
tipo_manej     178
diag_medic     159
gp_gestan       37
gp_mad_com      37
niv_educat      34
ocupacion_      13
clas_peso        9
clas_talla       7
gp_migrant       2
gp_pobicbf       2
gp_indigen       2
gp_desplaz       2
gp_vic_vio       2

 Tipos de datos:
anio                         int64
edad_                        int64
uni_med_                     int64
sexo_                       object
nacionali_                   int64
per_etn_                     int64
etnia                       object
nom_grupo_                  object
estrato_                     int64
cod_pais_o                   int64
pais_origen                 object
cod_dpto_o                  object
depto_origen                object
cod_mun_o                    int64
municipio_origen            object
area_                        int64
cod_pais_r                   int64
cod_

,anio,edad_,uni_med_,nacionali_,per_etn_,estrato_,cod_pais_o,cod_mun_o,area_,cod_pais_r,cod_mun_r,fec_not,semana,mes,fecha_nto_,fec_con_,niv_educat,menores,gp_desplaz,gp_migrant,gp_indigen,gp_pobicbf,gp_mad_com,gp_vic_vio,gp_gestan,ocupacion_,peso_nac,talla_nac,edad_ges,t_lechem,e_complem,crec_dllo,esq_vac,carne_vac,peso_act,talla_act,per_braqui,imc,zscore_pt,zscore_te,clas_peso,clas_talla,edema,delgadez,palidez,piel_rese,hiperpigm,cambios_cabello,ruta_atenc,tipo_manej,pac_hos_,con_fin_,tip_cas_,fuente_
count,2448.00,2448.00,2448.00,2448.00,2448.00,2448.00,2448.00,2448.00,2448.00,2448.00,2448.00,2448,2448.00,2448.00,2448,2448,2414.00,2448.00,2446.00,2446.00,2446.00,2446.00,2411.00,2446.00,2411.00,2435.00,2448.00,938.00,2448.00,2448.00,2448.00,2448.00,2448.00,2448.00,2448.00,2448.00,2220.00,2448.00,2448.00,2448.00,2439.00,2441.00,2448.00,2448.00,2448.00,2448.00,2448.00,2448.00,2448.00,2270.00,2448.00,2448.00,2448.00,2448.00
mean,2024.37,3.04,1.25,170.85,1.43,1.00,173.11,377.16,2.56,173.11,375.76,2024-11-13 04:00:00,26.63,6.55,2023-03-08 11:11:10.588235520,2024-11-11 21:03:31.764706048,3.15,1.31,2.00,2.00,2.00,1.97,2.00,2.00,2.00,98483.67,2769.21,48.41,38.46,9.83,5.74,1.46,1.81,1.63,7.38,72.97,12.31,13.59,-2.34,-2.78,2.08,1.38,1.81,1.29,1.59,1.41,1.64,1.38,1.06,1.43,1.59,1.00,4.00,1.22
min,2023.00,1.00,1.00,170.00,1.00,0.00,170.00,1.00,1.00,170.00,1.00,2023-01-10 00:00:00,1.00,1.00,2018-07-16 00:00:00,2023-01-10 00:00:00,1.00,0.00,1.00,1.00,1.00,1.00,2.00,1.00,2.00,9999.00,950.00,35.00,22.00,0.00,0.00,1.00,1.00,1.00,2.30,47.00,6.00,10.00,-5.96,-6.00,1.00,1.00,1.00,1.00,1.00,1.00,1.00,1.00,1.00,1.00,1.00,1.00,4.00,1.00
25%,2024.00,1.00,1.00,170.00,1.00,1.00,170.00,13.00,3.00,170.00,13.00,2024-06-01 00:00:00,15.00,4.00,2022-07-18 12:00:00,2024-06-01 00:00:00,1.00,1.00,2.00,2.00,2.00,2.00,2.00,2.00,2.00,99999.07,2737.50,47.00,38.00,6.00,6.00,1.00,1.00,1.00,6.20,67.00,11.60,13.00,-2.73,-3.76,2.00,1.00,2.00,1.00,1.00,1.00,1.00,1.00,1.00,1.00,1.00,1.00,4.00,1.00
50%,2024.00,2.00,1.00,170.00,1.00,1.00,170.00,430.00,3.00,170.00,430.00,2024-10-18 00:00:00,27.00,7.00,2023-05-18 12:00:00,2024-10-18 00:00:00,3.00,1.00,2.00,2.00,2.00,2.00,2.00,2.00,2.00,99999.07,2845.00,49.00,39.00,8.00,6.00,1.00,2.00,2.00,7.29,72.00,12.00,13.52,-2.32,-2.84,2.00,1.00,2.00,1.00,2.00,1.00,2.00,1.00,1.00,1.00,2.00,1.00,4.00,1.00
75%,2025.00,4.00,2.00,170.00,1.00,1.00,170.00,570.00,3.00,170.00,570.00,2025-05-26 06:00:00,39.00,9.00,2024-01-02 00:00:00,2025-05-26 00:00:00,5.00,2.00,2.00,2.00,2.00,2.00,2.00,2.00,2.00,99999.07,2845.00,50.00,39.00,12.00,6.00,2.00,3.00,2.00,8.40,78.00,13.00,14.10,-2.11,-1.89,2.00,2.00,2.00,2.00,2.00,2.00,2.00,2.00,1.00,2.00,2.00,1.00,4.00,1.00
max,2025.00,11.00,2.00,862.00,6.00,3.00,862.00,980.00,3.00,862.00,980.00,2026-01-05 00:00:00,53.00,12.00,2025-11-09 00:00:00,2025-12-30 00:00:00,5.00,11.00,2.00,2.00,2.00,2.00,2.00,2.00,2.00,99999.07,4900.00,55.00,42.00,99.00,99.00,2.00,3.00,2.00,18.50,110.00,18.50,21.90,2.95,4.73,6.00,3.00,2.00,2.00,2.00,2.00,2.00,2.00,2.00,2.00,2.00,1.00,4.00,4.00
std,0.56,3.02,0.43,24.21,1.40,0.20,46.29,334.03,0.79,46.29,333.06,NaN,14.40,3.30,NaN,NaN,1.84,1.13,0.06,0.06,0.02,0.16,0.00,0.02,0.00,11582.09,329.84,3.07,1.67,6.76,3.79,0.50,0.87,0.48,2.10,10.16,1.22,1.31,1.01,1.47,0.78,0.67,0.39,0.45,0.49,0.49,0.48,0.49,0.25,0.50,0.49,0.00,0.00,0.70



 Primeras 5 filas:


,anio,edad_,uni_med_,sexo_,nacionali_,per_etn_,etnia,nom_grupo_,estrato_,cod_pais_o,pais_origen,cod_dpto_o,depto_origen,cod_mun_o,municipio_origen,area_,cod_pais_r,cod_dpto_r,cod_mun_r,ndep_resi,nmun_resi,fec_not,semana,mes,anio_mes,fecha_nto_,fec_con_,tip_ss_,niv_educat,menores,gp_desplaz,gp_migrant,gp_indigen,gp_pobicbf,gp_mad_com,gp_vic_vio,gp_gestan,ocupacion_,peso_nac,talla_nac,edad_ges,t_lechem,e_complem,crec_dllo,esq_vac,carne_vac,peso_act,talla_act,per_braqui,imc,zscore_pt,zscore_te,clas_peso,clas_talla,edema,delgadez,palidez,piel_rese,hiperpigm,cambios_cabello,diag_medic,ruta_atenc,tipo_manej,pac_hos_,con_fin_,tip_cas_,fuente_
0,2023,6,2,M,170,1,Indigena,Wayuu,1,170,Colombia,44,La Guajira,847,Uribia,3,170,44,847,GUAJIRA,URIBIA,2023-01-10,2,1,2023-01,2022-06-29,2023-01-10,S,5.00,1.00,2.00,2.00,2.00,2.00,2.00,2.00,2.00,99999.07,2620.00,47.00,38.00,10.00,4.00,1,1,1,5.00,60.00,11.50,13.89,-2.24,-3.56,2,1,2,1,2,2,2,2,E440,1,2,2,1,4,1
1,2023,1,1,M,170,1,Indigena,Wayuu,1,170,Colombia,44,La Guajira,847,Uribia,3,170,44,847,GUAJIRA,URIBIA,2023-05-08,19,5,2023-05,2022-04-13,2023-05-08,S,1.00,2.00,2.00,2.00,2.00,2.00,2.00,2.00,2.00,99999.07,2800.00,NaN,38.00,6.00,6.00,1,1,1,7.40,71.50,12.10,14.50,-2.11,-1.79,2,2,2,1,2,2,2,1,E440,1,1,1,1,4,1
2,2023,10,2,M,170,1,Indigena,Arhuaco,1,170,Colombia,20,Cesar,570,Pueblo Bello,3,170,20,570,CESAR,PUEBLO BELLO,2023-10-04,40,10,2023-10,2022-11-24,2023-10-04,S,5.00,1.00,2.00,2.00,2.00,2.00,2.00,2.00,2.00,99999.07,3000.00,NaN,40.00,6.00,6.00,1,3,2,7.30,72.00,13.00,14.10,-2.44,-0.56,2,3,2,1,2,2,1,1,E440,1,2,2,1,4,1
3,2023,1,1,F,170,6,Otro,<NA>,1,170,Colombia,20,Cesar,570,Pueblo Bello,3,170,20,570,CESAR,PUEBLO BELLO,2023-05-11,19,5,2023-05,2022-03-11,2023-05-11,S,1.00,1.00,2.00,2.00,2.00,2.00,2.00,2.00,2.00,99999.07,2900.00,NaN,39.00,6.00,6.00,2,1,2,7.30,70.50,6.00,14.70,-1.40,-2.19,3,1,1,1,1,1,1,1,E440,1,1,1,1,4,1
4,2023,2,1,F,170,6,Otro,<NA>,2,170,Colombia,20,Cesar,570,Pueblo Bello,1,170,20,570,CESAR,PUEBLO BELLO,2023-03-08,10,3,2023-03,2021-02-16,2023-03-08,S,5.00,4.00,2.00,2.00,2.00,2.00,2.00,2.00,2.00,99999.07,2900.00,NaN,39.00,0.00,0.00,2,3,2,8.70,79.00,13.70,13.94,-1.63,-2.08,3,1,1,1,1,1,1,1,NaN,1,<NA>,1,1,4,1


## 22. Exportación de archivos de salida

| Archivo | Descripción | Notebook destino |
|---|---|---|
| `113_limpio_unificado.csv` | Dataset completo con etiquetas y fechas | 04 análisis exploratorio |
| `dataset_ml.csv` | Sin zscore leakage, sin fechas, para modelos | 05 modelos ML |
| `serie_temporal_mensual.csv` | Métricas agregadas por mes (36+ puntos) | 06 predicción temporal |
| `serie_temporal_departamento.csv` | Comparativa por departamento y año | 07 comparativa territorial |

> **Nota sobre `zscore_pt` y `zscore_te` en `dataset_ml.csv`:** Se excluyen del dataset de ML
> porque son derivadas directas de `clas_peso` y `clas_talla` (Resolución 2465 de 2016).
> Incluirlas haría que el modelo aprenda a replicar una fórmula matemática, no a identificar
> riesgo real con variables observables.

In [59]:
import os
os.makedirs("../data/processed", exist_ok=True)

# 1. Dataset completo (con etiquetas, fechas y zscore para EDA)
ruta_limpio = "../data/processed/113_limpio_unificado.csv"
data_unificado.to_csv(ruta_limpio, index=False)
print(f"Dataset limpio guardado en: {ruta_limpio}")
print(f"Shape: {data_unificado.shape}")

Dataset limpio guardado en: ../data/processed/113_limpio_unificado.csv
Shape: (2448, 67)


In [60]:
# 2. Dataset ML (sin leakage, sin fechas, sin columnas de texto libre)
cols_excluir_ml = [
    "fec_not", "fecha_nto_", "fec_con_", "FechaHora",
    "anio_mes", "mes", "semana",
    "nombre_nacionalidad", "nom_grupo_", "diag_medic",
    # DATA LEAKAGE: zscore define directamente clas_peso/clas_talla (Res. 2465/2016)
    "zscore_pt", "zscore_te",
    # Etiquetas derivadas (redundantes con los codigos numericos)
    "etnia", "pais_origen", "depto_origen", "municipio_origen",
]
data_ml = data_unificado.drop(
    columns=[c for c in cols_excluir_ml if c in data_unificado.columns]
)
data_ml.to_csv("../data/processed/dataset_ml.csv", index=False)
print(f"✅ dataset_ml.csv               — {data_ml.shape[0]:,} filas × {data_ml.shape[1]} cols")

# 3. Serie temporal mensual
serie_mensual = (
    data_unificado
    .groupby("anio_mes")
    .agg(
        casos              = ("clas_peso", "count"),
        tasa_desnut        = ("clas_peso", lambda x: (x.isin([1, 2])).mean() * 100),
        tasa_severa        = ("clas_peso", lambda x: (x == 1).mean() * 100),
        tasa_moderada      = ("clas_peso", lambda x: (x == 2).mean() * 100),
        zscore_pt_media    = ("zscore_pt", "mean"),
        zscore_pt_std      = ("zscore_pt", "std"),
        peso_act_media     = ("peso_act", "mean"),
        imc_media          = ("imc", "mean"),
        pct_rural          = ("area_", lambda x: (x == 3).mean() * 100),
        pct_sin_seguim     = ("crec_dllo", lambda x: (x == 2).mean() * 100),
        pct_vac_incompleta = ("esq_vac", lambda x: (x.isin([2, 3])).mean() * 100),
    )
    .reset_index()
    .sort_values("anio_mes")
)
serie_mensual.to_csv("../data/processed/serie_temporal_mensual.csv", index=False)
print(f"✅ serie_temporal_mensual.csv   — {len(serie_mensual)} puntos mensuales")

# 4. Serie por departamento
serie_dpto = (
    data_unificado[data_unificado["ndep_resi"].notna()]
    .groupby(["anio", "ndep_resi"])
    .agg(
        casos        = ("clas_peso", "count"),
        tasa_desnut  = ("clas_peso", lambda x: (x.isin([1, 2])).mean() * 100),
        tasa_severa  = ("clas_peso", lambda x: (x == 1).mean() * 100),
        zscore_medio = ("zscore_pt", "mean"),
        peso_act_med = ("peso_act", "mean"),
    )
    .reset_index()
    .rename(columns={"ndep_resi": "departamento"})
    .sort_values(["anio", "departamento"])
)
serie_dpto.to_csv("../data/processed/serie_temporal_departamento.csv", index=False)
print(f"✅ serie_temporal_dpto.csv      — {len(serie_dpto)} filas")

print("\n" + "="*60)
print("  ETL COMPLETADO EXITOSAMENTE")
print("="*60)
print(f"  Registros limpios : {data_unificado.shape[0]:,}")
print(f"  Columnas completo : {data_unificado.shape[1]}")
print(f"  Columnas ML       : {data_ml.shape[1]}")
print(f"  Puntos temporales : {len(serie_mensual)} meses")
print(f"  Rango temporal    : {serie_mensual['anio_mes'].min()} → {serie_mensual['anio_mes'].max()}")
print("\n  Siguiente paso    : 04_analisis_exploratorio.ipynb")
print("="*60)

✅ dataset_ml.csv               — 2,448 filas × 53 cols
✅ serie_temporal_mensual.csv   — 37 puntos mensuales
✅ serie_temporal_dpto.csv      — 19 filas

  ETL COMPLETADO EXITOSAMENTE
  Registros limpios : 2,448
  Columnas completo : 67
  Columnas ML       : 53
  Puntos temporales : 37 meses
  Rango temporal    : 2023-01 → 2026-01

  Siguiente paso    : 04_analisis_exploratorio.ipynb
